In [ ]:
import torch
import torch.optim as optim
import gymnasium
import numpy as np

from RLAlg.buffer.rollout_buffer import RolloutBuffer
from RLAlg.alg.ppo import PPO
from model import GaussianActor, Critic

In [ ]:
def setup_env(env_name, mode=None):
    env = gymnasium.make(env_name, render_mode=mode)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)

    return env

In [ ]:
env_name = "Pendulum-v1"
env_num = 10
max_steps = 200

envs = gymnasium.vector.SyncVectorEnv([lambda: setup_env(env_name) for _ in range(env_num)])

obs_space = np.prod(envs.single_observation_space.shape)
action_space = np.prod(envs.single_action_space.shape)

actor = GaussianActor(obs_space, action_space, [128, 128])
critic = Critic(obs_space, [128, 128])

optimizer = optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=3e-4)

buffer = RolloutBuffer(env_num, max_steps, envs.single_observation_space.shape, envs.single_action_space.shape)

In [ ]:
def average_non_zero(numbers):
    non_zero_numbers = [num for num in numbers if num != 0]
    if not non_zero_numbers:
        return 0  # Return 0 if there are no non-zero elements
    return sum(non_zero_numbers) / len(non_zero_numbers)

In [ ]:
@torch.no_grad()
def get_action(state):
    state = torch.as_tensor(state).float()
    pi, action, log_prob = actor(state)
    value = critic(state)
    return action.tolist(), log_prob.tolist(), value.tolist()

In [ ]:
def rollout():
    state, info = envs.reset()
    for i in range(200):
        action, log_prob, value = get_action(state)
        next_state, reward, done, timeout, info = envs.step(action)
        buffer.add_steps(i, state, action, log_prob, reward, done, value)
        state = next_state
        
        if "episode" in info:
            print(average_non_zero(info['episode']['r']))
            
    _, _, value = get_action(state)        
    buffer.compute_gae(value, 0.99, 0.95)

In [ ]:
def update():
    for _ in range(10):
        for batch in buffer.batch_sample(200):
            states = batch["states"]
            actions = batch["actions"]
            log_probs = batch["log_probs"]
            values = batch["values"]
            returns = batch["returns"]
            advantages = batch["advantages"]
            
            actor_loss, entropy_loss = PPO.compute_policy_loss(actor, log_probs, states, actions, advantages, 0.2)
            critic_loss = PPO.compute_value_loss(critic, states, returns)
            
            loss = actor_loss + 0.5 * critic_loss + 0.01 * entropy_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [ ]:
for _ in range(100):
    rollout()
    update()